In [1]:
import numpy as np
from engin_core import fit_gp, simulate_unit
from engin_core.tea import CostParameters, cost_summary

rng = np.random.default_rng(7)
U = rng.random((40, 5))                       # 40-run DoE over the process knobs
y = simulate_unit(U) + rng.normal(0, 1.5, 40)  # with assay noise
gp = fit_gp(U, y, seed=0)

best = int(np.argmax(y))
c = cost_summary(gp, U[[best]])[0]
print(f"expected   ${c.expected_usd_per_kg:.2f}/kg")
print(f"90% range  ${c.lower_usd_per_kg:.2f} - ${c.upper_usd_per_kg:.2f}")
print(f"P(clears ${c.target_usd_per_kg:.0f}/kg) = {c.prob_meets_target:.2f}")

expected   $57.19/kg
90% range  $55.16 - $59.31
P(clears $200/kg) = 1.00


In [2]:
from engin_core.tea import break_even

for target in (200.0, 40.0, 5.0):
    be = break_even(U[[best]], params=CostParameters(target_usd_per_kg=target), gp=gp)
    titer = f"{be.value:.1f} g/L" if be.value is not None else "unreachable"
    reach = f"{be.prob_reaching:.2f}" if be.prob_reaching is not None else "n/a"
    print(f"  ${target:>6.0f}/kg  ->  {titer:>14}   reachable={be.reachable!s:<5} P(reach)={reach}")

  $   200/kg  ->        17.9 g/L   reachable=True  P(reach)=1.00
  $    40/kg  ->       137.6 g/L   reachable=True  P(reach)=0.00
  $     5/kg  ->     unreachable   reachable=False P(reach)=n/a


In [3]:
mean, sd = gp.predict(U)
summaries = cost_summary(gp, U)
width = np.array([s.upper_usd_per_kg - s.lower_usd_per_kg for s in summaries])

print(f"  corr(interval width, GP sd)   = {np.corrcoef(width, sd)[0, 1]:+.2f}")
print(f"  corr(interval width, 1/titer) = {np.corrcoef(width, 1 / mean)[0, 1]:+.2f}")
print()
for i in np.argsort(mean)[:: max(len(mean) // 4, 1)][:4]:
    s = summaries[i]
    w = s.upper_usd_per_kg - s.lower_usd_per_kg
    print(f"  titer {mean[i]:5.1f} g/L   cost ${s.expected_usd_per_kg:7.2f}/kg   "
          f"90% width ${w:6.2f}  ({w / s.expected_usd_per_kg:.0%} of the estimate)")

  corr(interval width, GP sd)   = +0.32
  corr(interval width, 1/titer) = +0.95

  titer   9.3 g/L   cost $ 362.76/kg   90% width $298.51  (82% of the estimate)
  titer  28.1 g/L   cost $ 135.33/kg   90% width $ 27.00  (20% of the estimate)
  titer  43.7 g/L   cost $  95.40/kg   90% width $ 11.91  (12% of the estimate)
  titer  58.2 g/L   cost $  76.16/kg   90% width $  7.36  (10% of the estimate)


In [4]:
from engin_core.tea import ParametricCostModel

model = ParametricCostModel()
bd = model.cost_breakdown(np.array([mean[best]]), U[[best]])
total = sum(float(np.atleast_1d(v)[0]) for k, v in bd.items() if k != "total")
for name, value in bd.items():
    v = float(np.atleast_1d(value)[0])
    share = "" if name == "total" else f"   ({v / total:.1%})"
    print(f"  {name:<14} ${v:>8.2f}/kg{share}")

  raw_material   $    1.57/kg   (2.8%)
  facility       $   25.26/kg   (44.2%)
  downstream     $   30.29/kg   (53.0%)
  capital        $    0.00/kg   (0.0%)


In [5]:
from engin_core.simulator import ReactorConfig
from engin_core.tea import PurityGrade, purity_dsp_multiplier

params = CostParameters(
    substrate_usd_per_kg=0.55,          # what you pay for feedstock
    target_usd_per_kg=200.0,            # the price you need to beat
)
vessel = ReactorConfig(v0=1.0, vmax=2.5, t_end=48.0)

for grade in PurityGrade:
    print(f"  {grade.value:<10} downstream multiplier x{purity_dsp_multiplier(grade):.2f}")

  crude      downstream multiplier x1.00
  purified   downstream multiplier x3.32
